# Assignment 9 — Conv-VAE for Compression & Denoising

My own version: a **convolutional** VAE on MNIST (not an MLP), trained 8 epochs each for latent sizes 2 / 16 / 32.

- Part A: compression 784 -> z
- Part B: rebuild 10 test digits (one of each 0-9), MSE + PSNR
- Part C: heavy Gaussian noise (std 0.6), then denoise
- Part D: bigger latent = sharper rebuilds

Run: Colab GPU > Run all. ~2 min per latent size.

In [1]:
# Setup — my own choices (seed 123, batch 256, lr 2e-3)
import random, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 123
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(torch.__version__, device)

BATCH, EPOCHS = 256, 8
Z_LIST = [2, 16, 32]
SIGMA = 0.6  # stronger noise than usual, so denoising is visible

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Data
tr = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
te = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(tr, batch_size=BATCH, shuffle=True, num_workers=2)
print(len(tr), len(te))

# 10 test digits: first occurrence of each digit 0-9 (not just indices 0-9)
need = list(range(10)); picks, seen = [], set()
for idx in range(len(te)):
    _, y = te[idx]
    if y not in seen:
        picks.append(idx); seen.add(y)
    if len(picks) == 10: break
order = sorted(range(10), key=lambda k: te[picks[k]][1])  # sort 0..9
picks = [picks[k] for k in order]
ten = torch.stack([te[i][0] for i in picks]).to(device)
ten_labels = [te[i][1] for i in picks]
print('test indices:', picks)
print('labels:', ten_labels)

In [ ]:
# Conv-VAE: conv encoder + deconv decoder (different from MLP VAEs)
class ConvVAE(nn.Module):
    def __init__(self, z=16):
        super().__init__()
        self.z = z
        self.e1 = nn.Conv2d(1, 16, 3, 2, 1)    # 28 -> 14
        self.e2 = nn.Conv2d(16, 32, 3, 2, 1)   # 14 -> 7
        self.eh = nn.Linear(32*7*7, 256)
        self.mu = nn.Linear(256, z)
        self.lv = nn.Linear(256, z)
        self.dh = nn.Linear(z, 256)
        self.du = nn.Linear(256, 32*7*7)
        self.d1 = nn.ConvTranspose2d(32, 16, 4, 2, 1)  # 7 -> 14
        self.d2 = nn.ConvTranspose2d(16, 1, 4, 2, 1)   # 14 -> 28
    def encode(self, x):
        h = F.relu(self.e1(x)); h = F.relu(self.e2(h))
        h = F.relu(self.eh(h.flatten(1)))
        return self.mu(h), self.lv(h)
    def decode(self, zv):
        h = F.relu(self.dh(zv)); h = F.relu(self.du(h)).view(-1, 32, 7, 7)
        h = F.relu(self.d1(h))
        return torch.sigmoid(self.d2(h))
    def forward(self, x):
        m, l = self.encode(x)
        z = m + torch.randn_like(m) * torch.exp(0.5*l)
        return self.decode(z), m, l

# MSE + KL (per-image mean). Reference notebooks often use BCE-sum; MSE keeps denoising fair.
def loss_fn(r, x, m, l):
    rec = F.mse_loss(r, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + l - m.pow(2) - l.exp()) / x.size(0)
    return rec + kl, rec, kl

def psnr(mse):
    return float(10*np.log10(1.0/max(mse, 1e-12)))

m0 = ConvVAE(z=16)
print('params:', sum(p.numel() for p in m0.parameters()))
with torch.no_grad():
    r, _, _ = m0(torch.rand(4, 1, 28, 28))
print('io:', tuple(r.shape))

In [ ]:
# Train one Conv-VAE
def fit(z):
    torch.manual_seed(SEED)
    net = ConvVAE(z=z).to(device)
    opt = optim.Adam(net.parameters(), lr=2e-3)
    hist = []
    for ep in range(1, EPOCHS+1):
        net.train(); tot = 0
        for x, _ in train_loader:
            x = x.to(device)
            r, m, l = net(x)
            loss, _, _ = loss_fn(r, x, m, l)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*x.size(0)
        avg = tot/len(train_loader.dataset)
        hist.append(avg); print(f'z={z} epoch {ep}/{EPOCHS} loss {avg:.4f}')
    torch.save(net.state_dict(), f'convvae_z{z}.pt')
    return net, hist

nets, hists = {}, {}
for z in Z_LIST:
    nets[z], hists[z] = fit(z)
plt.figure()
for z in Z_LIST: plt.plot(hists[z], label=f'z={z}')
plt.legend(); plt.xlabel('epoch'); plt.ylabel('loss/img'); plt.title('Training loss')
plt.savefig('conv_loss.png'); plt.show()

## Part A — Compression

In [ ]:
for z in Z_LIST:
    print(f'z={z}: 784 -> {z}  = {784/z:.1f}x smaller ({784*4} bytes -> {z*4} bytes float32)')

## Part B — Rebuild 10 digits (0-9), MSE + PSNR

In [ ]:
rec_mse, rec_psnr = {}, {}
with torch.no_grad():
    for z in Z_LIST:
        nets[z].eval()
        r, _, _ = nets[z](ten)
        e = ((ten - r)**2).mean(dim=(1,2,3)).cpu().numpy()
        rec_mse[z] = float(e.mean()); rec_psnr[z] = psnr(rec_mse[z])
        print(f'z={z} mean MSE {rec_mse[z]:.4f}  PSNR {rec_psnr[z]:.2f} dB')

# one figure: 4 rows x 10 cols (orig + 3 reconstructions)
fig, ax = plt.subplots(4, 10, figsize=(12, 5))
with torch.no_grad():
    outs = {z: nets[z](ten)[0] for z in Z_LIST}
for j in range(10):
    ax[0, j].imshow(ten[j].cpu().squeeze(), cmap='gray'); ax[0, j].set_title(f'{ten_labels[j]}'); ax[0, j].axis('off')
    for i, z in enumerate(Z_LIST):
        ax[i+1, j].imshow(outs[z][j].cpu().squeeze(), cmap='gray'); ax[i+1, j].axis('off')
ax[0, 0].set_ylabel('orig'); ax[1, 0].set_ylabel('z=2'); ax[2, 0].set_ylabel('z=16'); ax[3, 0].set_ylabel('z=32')
plt.suptitle('Row1=original, rows2-4=rebuilds z=2/16/32')
plt.tight_layout(); plt.savefig('conv_reconstruction.png'); plt.show()

## Part C — Denoise heavy noise (std 0.6)

In [ ]:
g = torch.Generator(device=device).manual_seed(0)
noise = torch.randn(ten.shape, generator=g, device=device) * SIGMA
noisy = torch.clamp(ten + noise, 0, 1)
base = float(((ten - noisy)**2).mean())
print(f'noisy MSE without VAE: {base:.4f}')
den_mse = {}
with torch.no_grad():
    for z in Z_LIST:
        d = nets[z].decode(nets[z].encode(noisy)[0])  # deterministic: use mean
        den_mse[z] = float(((ten - d)**2).mean())
        print(f'z={z} denoised MSE {den_mse[z]:.4f} (gain {(1-den_mse[z]/base)*100:.1f}%)')
    best = nets[32].decode(nets[32].encode(noisy)[0])

fig, ax = plt.subplots(3, 10, figsize=(12, 4))
for j in range(10):
    ax[0, j].imshow(ten[j].cpu().squeeze(), cmap='gray'); ax[0, j].axis('off')
    ax[1, j].imshow(noisy[j].cpu().squeeze(), cmap='gray'); ax[1, j].axis('off')
    ax[2, j].imshow(best[j].cpu().squeeze(), cmap='gray'); ax[2, j].axis('off')
ax[0, 0].set_ylabel('clean'); ax[1, 0].set_ylabel('noisy'); ax[2, 0].set_ylabel('fixed z=32')
plt.suptitle(f'Denoising sigma={SIGMA}')
plt.tight_layout(); plt.savefig('conv_denoising.png'); plt.show()

# extra: sweep noise levels with z=32
with torch.no_grad():
    nets[32].eval()
    sigs, raws, fixs = [0.2, 0.4, 0.6, 0.8], [], []
    for s in sigs:
        n = torch.clamp(ten + torch.randn_like(ten)*s, 0, 1)
        f = nets[32].decode(nets[32].encode(n)[0])
        raws.append(float(((ten-n)**2).mean())); fixs.append(float(((ten-f)**2).mean()))
plt.figure(); plt.plot(sigs, raws, 'o-', label='noisy'); plt.plot(sigs, fixs, 's-', label='denoised z=32')
plt.xlabel('noise std'); plt.ylabel('MSE'); plt.legend(); plt.title('Denoising over noise levels')
plt.savefig('conv_sweep.png'); plt.show()

## Part D + results table

In [ ]:
# Latent-space scatter for z=2 (1000 test points, coloured by digit)
import torch.utils.data as tud
sub = tud.Subset(te, list(range(1000)))
dl = tud.DataLoader(sub, batch_size=500)
zs, ys = [], []
nets[2].eval()
with torch.no_grad():
    for x, y in dl:
        m, _ = nets[2].encode(x.to(device))
        zs.append(m.cpu()); ys.append(y)
zs = torch.cat(zs).numpy(); ys = torch.cat(ys).numpy()
plt.figure(figsize=(4,4)); sc = plt.scatter(zs[:,0], zs[:,1], c=ys, cmap='tab10', s=5)
plt.colorbar(sc, label='digit'); plt.title('Latent space z=2 (1000 points)')
plt.savefig('conv_latent2d.png'); plt.show()

rows = []
for z in Z_LIST:
    note = 'fuzzy blobs' if z==2 else ('good shapes' if z==16 else 'sharpest')
    rows.append({'z': z, 'stored': z, 'ratio': f'{784/z:.1f}x',
                 'recon_MSE': round(rec_mse[z],4), 'recon_PSNR_dB': round(rec_psnr[z],2),
                 'denoise_MSE': round(den_mse[z],4), 'looks': note})
df = pd.DataFrame(rows)
display(df); df.to_csv('conv_results.csv', index=False)
print('saved conv_reconstruction.png conv_denoising.png conv_sweep.png conv_latent2d.png conv_results.csv')